## Osadzenia

W naszym poprzednim przykładzie operowaliśmy na wektorach torby słów o wysokiej wymiarowości i długości `vocab_size`, i explicite konwertowaliśmy niskowymiarowe wektory reprezentacji pozycyjnej na rzadką reprezentację one-hot. Ta reprezentacja one-hot nie jest efektywna pamięciowo, dodatkowo każde słowo jest traktowane niezależnie od innych, tzn. wektory zakodowane one-hot nie wyrażają żadnego semantycznego podobieństwa między słowami.

W tej części będziemy kontynuować eksplorację zbioru danych **News AG**. Na początek załadujmy dane i pobierzmy niektóre definicje z poprzedniego notatnika.


In [1]:
import torch
import torchtext
import numpy as np
from torchnlp import *
train_dataset, test_dataset, classes, vocab = load_dataset()
vocab_size = len(vocab)
print("Vocab size = ",vocab_size)

Loading dataset...


d:\WORK\ai-for-beginners\5-NLP\14-Embeddings\data\train.csv: 29.5MB [00:01, 18.8MB/s]                            
d:\WORK\ai-for-beginners\5-NLP\14-Embeddings\data\test.csv: 1.86MB [00:00, 11.2MB/s]                          


Building vocab...
Vocab size =  95812



## Co to jest embedding?

Idea **embeddingu** polega na reprezentowaniu słów za pomocą gęstych wektorów o niższej wymiarowości, które w pewien sposób odzwierciedlają semantyczne znaczenie słowa. Później omówimy, jak tworzyć sensowne embeddingi słów, ale na razie pomyślmy o embeddingach jako o sposobie na zmniejszenie wymiarowości wektora słowa. 

Warstwa embeddingu przyjmuje słowo jako dane wejściowe i generuje wektor wyjściowy o określonym `embedding_size`. W pewnym sensie jest to bardzo podobne do warstwy `Linear`, ale zamiast przyjmować wektor zakodowany metodą jeden-na-jeden, potrafi przyjąć numer słowa jako wejście.

Korzystając z warstwy embeddingu jako pierwszej warstwy w naszej sieci, możemy przejść od modelu worek słów do modelu **embedding bag**, w którym najpierw przekształcamy każde słowo w naszym tekście na odpowiadający embedding, a następnie obliczamy jakąś funkcję agregującą na wszystkich tych embeddingach, taką jak `sum`, `average` lub `max`.  

![Obraz pokazujący klasyfikator embeddingu dla pięciu słów w sekwencji.](../../../../../translated_images/pl/embedding-classifier-example.b77f021a7ee67eee.webp)

Nasza sieć neuronowa klasyfikatora zacznie się od warstwy embeddingu, potem warstwa agregacji, a na jej szczycie klasyfikator liniowy:


In [2]:
class EmbedClassifier(torch.nn.Module):
    def __init__(self, vocab_size, embed_dim, num_class):
        super().__init__()
        self.embedding = torch.nn.Embedding(vocab_size, embed_dim)
        self.fc = torch.nn.Linear(embed_dim, num_class)

    def forward(self, x):
        x = self.embedding(x)
        x = torch.mean(x,dim=1)
        return self.fc(x)

### Radzenie sobie ze zmiennym rozmiarem sekwencji

W wyniku takiej architektury minibatche do naszej sieci musiałyby być tworzone w określony sposób. W poprzedniej jednostce, korzystając z bag-of-words, wszystkie tensory BoW w minibatchu miały ten sam rozmiar `vocab_size`, niezależnie od faktycznej długości naszej sekwencji tekstowej. Po przejściu na osadzenia słów mielibyśmy zmienną liczbę słów w każdej próbce tekstu, a podczas łączenia tych próbek w minibatche musielibyśmy zastosować padding.

Można to zrobić używając tej samej techniki dostarczania funkcji `collate_fn` do źródła danych:


In [3]:
def padify(b):
    # b is the list of tuples of length batch_size
    #   - first element of a tuple = label, 
    #   - second = feature (text sequence)
    # build vectorized sequence
    v = [encode(x[1]) for x in b]
    # first, compute max length of a sequence in this minibatch
    l = max(map(len,v))
    return ( # tuple of two tensors - labels and features
        torch.LongTensor([t[0]-1 for t in b]),
        torch.stack([torch.nn.functional.pad(torch.tensor(t),(0,l-len(t)),mode='constant',value=0) for t in v])
    )

train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=16, collate_fn=padify, shuffle=True)

### Trenowanie klasyfikatora osadzania

Teraz, gdy zdefiniowaliśmy odpowiedni dataloader, możemy trenować model za pomocą funkcji treningowej, którą zdefiniowaliśmy w poprzedniej jednostce:


In [4]:
net = EmbedClassifier(vocab_size,32,len(classes)).to(device)
train_epoch(net,train_loader, lr=1, epoch_size=25000)

3200: acc=0.6415625
6400: acc=0.6865625
9600: acc=0.7103125
12800: acc=0.726953125
16000: acc=0.739375
19200: acc=0.75046875
22400: acc=0.7572321428571429


(0.889799795315499, 0.7623160588611644)

> **Uwaga**: Trenujemy tutaj tylko na 25 tysiącach rekordów (mniej niż pełna epoka) ze względu na czas, ale możesz kontynuować trening, napisać funkcję do trenowania przez kilka epok i eksperymentować z parametrem współczynnika uczenia, aby osiągnąć wyższą dokładność. Powinieneś być w stanie osiągnąć dokładność około 90%.


### Warstwa EmbeddingBag i reprezentacja sekwencji o zmiennej długości

W poprzedniej architekturze musieliśmy dopełnić wszystkie sekwencje do tej samej długości, aby zmieścić je w minibatchu. Nie jest to najbardziej efektywny sposób reprezentacji sekwencji o zmiennej długości - innym podejściem byłoby użycie wektora **offsetów**, który przechowywałby przesunięcia wszystkich sekwencji zapisanych w jednym dużym wektorze.

![Obraz przedstawiający reprezentację sekwencji za pomocą offsetów](../../../../../translated_images/pl/offset-sequence-representation.eb73fcefb29b46ee.webp)

> **Uwaga**: Na powyższym obrazku pokazujemy sekwencję znaków, ale w naszym przykładzie pracujemy z sekwencjami słów. Jednak ogólna zasada reprezentowania sekwencji za pomocą wektora offsetów pozostaje taka sama.

Do pracy z reprezentacją offsetową używamy warstwy [`EmbeddingBag`](https://pytorch.org/docs/stable/generated/torch.nn.EmbeddingBag.html). Jest ona podobna do `Embedding`, ale przyjmuje jako wejście wektor zawartości oraz wektor offsetów, a także zawiera warstwę uśredniającą, którą może być `mean`, `sum` lub `max`.

Poniżej znajduje się zmodyfikowana sieć używająca `EmbeddingBag`:


In [5]:
class EmbedClassifier(torch.nn.Module):
    def __init__(self, vocab_size, embed_dim, num_class):
        super().__init__()
        self.embedding = torch.nn.EmbeddingBag(vocab_size, embed_dim)
        self.fc = torch.nn.Linear(embed_dim, num_class)

    def forward(self, text, off):
        x = self.embedding(text, off)
        return self.fc(x)

Aby przygotować zestaw danych do treningu, musimy dostarczyć funkcję konwersji, która przygotuje wektor przesunięcia:


In [6]:
def offsetify(b):
    # first, compute data tensor from all sequences
    x = [torch.tensor(encode(t[1])) for t in b]
    # now, compute the offsets by accumulating the tensor of sequence lengths
    o = [0] + [len(t) for t in x]
    o = torch.tensor(o[:-1]).cumsum(dim=0)
    return ( 
        torch.LongTensor([t[0]-1 for t in b]), # labels
        torch.cat(x), # text 
        o
    )

train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=16, collate_fn=offsetify, shuffle=True)

Zauważ, że w przeciwieństwie do wszystkich poprzednich przykładów, nasza sieć teraz akceptuje dwa parametry: wektor danych i wektor przesunięcia, które mają różne rozmiary. Podobnie, nasz loader danych dostarcza nam teraz 3 wartości zamiast 2: zarówno wektory tekstu, jak i przesunięć są dostarczane jako cechy. W związku z tym musimy nieco dostosować naszą funkcję treningową, aby się tym zająć:


In [7]:
net = EmbedClassifier(vocab_size,32,len(classes)).to(device)

def train_epoch_emb(net,dataloader,lr=0.01,optimizer=None,loss_fn = torch.nn.CrossEntropyLoss(),epoch_size=None, report_freq=200):
    optimizer = optimizer or torch.optim.Adam(net.parameters(),lr=lr)
    loss_fn = loss_fn.to(device)
    net.train()
    total_loss,acc,count,i = 0,0,0,0
    for labels,text,off in dataloader:
        optimizer.zero_grad()
        labels,text,off = labels.to(device), text.to(device), off.to(device)
        out = net(text, off)
        loss = loss_fn(out,labels) #cross_entropy(out,labels)
        loss.backward()
        optimizer.step()
        total_loss+=loss
        _,predicted = torch.max(out,1)
        acc+=(predicted==labels).sum()
        count+=len(labels)
        i+=1
        if i%report_freq==0:
            print(f"{count}: acc={acc.item()/count}")
        if epoch_size and count>epoch_size:
            break
    return total_loss.item()/count, acc.item()/count


train_epoch_emb(net,train_loader, lr=4, epoch_size=25000)

3200: acc=0.6153125
6400: acc=0.6615625
9600: acc=0.6932291666666667
12800: acc=0.715078125
16000: acc=0.7270625
19200: acc=0.7382291666666667
22400: acc=0.7486160714285715


(22.771553103007037, 0.7551983365323096)


## Osadzenia semantyczne: Word2Vec

W naszym poprzednim przykładzie warstwa osadzająca modelu nauczyła się mapować słowa na reprezentację wektorową, jednak ta reprezentacja nie miała zbyt dużego znaczenia semantycznego. Byłoby dobrze nauczyć się takiej reprezentacji wektorowej, że podobne słowa lub synonimy odpowiadałyby wektorom bliskim sobie pod względem pewnej metryki odległości wektorowej (np. odległości euklidesowej).

Aby to osiągnąć, musimy wstępnie wytrenować nasz model osadzający na dużym zbiorze tekstów w określony sposób. Jednym z pierwszych sposobów trenowania osadzeń semantycznych jest metoda zwana [Word2Vec](https://en.wikipedia.org/wiki/Word2vec). Opiera się ona na dwóch głównych architekturach służących do tworzenia rozproszonych reprezentacji słów:

 - **Continuous bag-of-words** (CBoW) — w tej architekturze trenujemy model, aby przewidywał słowo na podstawie otaczającego kontekstu. Mając n-gram $(W_{-2},W_{-1},W_0,W_1,W_2)$, celem modelu jest przewidzenie $W_0$ na podstawie $(W_{-2},W_{-1},W_1,W_2)$.
 - **Continuous skip-gram** jest przeciwieństwem CBoW. Model wykorzystuje otaczające słowa z okna kontekstu, aby przewidzieć słowo bieżące.

CBoW jest szybszy, podczas gdy skip-gram działa wolniej, ale lepiej reprezentuje rzadkie słowa.

![Obraz pokazujący oba algorytmy CBoW i Skip-Gram do konwersji słów na wektory.](../../../../../translated_images/pl/example-algorithms-for-converting-words-to-vectors.fbe9207a726922f6.webp)

Aby eksperymentować z embeddingiem word2vec wytrenowanym na zbiorze danych Google News, możemy użyć biblioteki **gensim**. Poniżej znajdują się słowa najbardziej podobne do 'neural'

> **Uwaga:** Przy pierwszym tworzeniu wektorów słów ich pobranie może trochę potrwać!


In [8]:
import gensim.downloader as api
w2v = api.load('word2vec-google-news-300')

In [9]:
for w,p in w2v.most_similar('neural'):
    print(f"{w} -> {p}")

neuronal -> 0.7804799675941467
neurons -> 0.7326500415802002
neural_circuits -> 0.7252851724624634
neuron -> 0.7174385190010071
cortical -> 0.6941086649894714
brain_circuitry -> 0.6923246383666992
synaptic -> 0.6699118614196777
neural_circuitry -> 0.6638563275337219
neurochemical -> 0.6555314064025879
neuronal_activity -> 0.6531826257705688


Możemy także obliczyć osadzenia wektorowe dla słowa, które będą używane do trenowania modelu klasyfikacji (dla przejrzystości pokazujemy tylko pierwsze 20 składowych wektora):


In [10]:
w2v.word_vec('play')[:20]

array([ 0.01226807,  0.06225586,  0.10693359,  0.05810547,  0.23828125,
        0.03686523,  0.05151367, -0.20703125,  0.01989746,  0.10058594,
       -0.03759766, -0.1015625 , -0.15820312, -0.08105469, -0.0390625 ,
       -0.05053711,  0.16015625,  0.2578125 ,  0.10058594, -0.25976562],
      dtype=float32)

Świetną cechą osadzeń semantycznych jest to, że można manipulować kodowaniem wektorowym, aby zmienić semantykę. Na przykład, możemy poprosić o znalezienie słowa, którego reprezentacja wektorowa byłaby możliwie jak najbliższa słowom *king* i *woman*, a jednocześnie jak najdalej od słowa *man*:


In [10]:
w2v.most_similar(positive=['king','woman'],negative=['man'])[0]

('queen', 0.7118192911148071)

Zarówno CBoW, jak i Skip-Grams to „predykcyjne” osadzenia, ponieważ uwzględniają jedynie lokalne konteksty. Word2Vec nie korzysta z globalnego kontekstu. 

**FastText** rozwija Word2Vec, ucząc reprezentacje wektorowe dla każdego słowa oraz n-gramów znakowych występujących w każdym słowie. Wartości reprezentacji są następnie uśredniane do jednego wektora na każdym etapie treningu. Choć dodaje to sporo dodatkowych obliczeń do pre-treningu, umożliwia kodowanie informacji podsłownej w osadzeniach słów. 

Inną metodą jest **GloVe**, która wykorzystuje ideę macierzy współwystępowania i stosuje metody neuronowe do rozkładu macierzy współwystępowania na bardziej wyraziste i nieliniowe wektory słów.

Możesz eksperymentować z przykładem, zmieniając osadzenia na FastText i GloVe, ponieważ gensim obsługuje kilka różnych modeli osadzeń słów.


## Używanie wstępnie wytrenowanych embeddingów w PyTorch

Możemy zmodyfikować powyższy przykład, aby wstępnie wypełnić macierz w naszej warstwie embeddingowej embeddingami semantycznymi, takimi jak Word2Vec. Musimy wziąć pod uwagę, że słowniki wstępnie wytrenowanych embeddingów i naszego korpusu tekstowego prawdopodobnie nie będą się pokrywać, więc zainicjujemy wagi dla brakujących słów losowymi wartościami:


In [11]:
embed_size = len(w2v.get_vector('hello'))
print(f'Embedding size: {embed_size}')

net = EmbedClassifier(vocab_size,embed_size,len(classes))

print('Populating matrix, this will take some time...',end='')
found, not_found = 0,0
for i,w in enumerate(vocab.get_itos()):
    try:
        net.embedding.weight[i].data = torch.tensor(w2v.get_vector(w))
        found+=1
    except:
        net.embedding.weight[i].data = torch.normal(0.0,1.0,(embed_size,))
        not_found+=1

print(f"Done, found {found} words, {not_found} words missing")
net = net.to(device)

Embedding size: 300
Populating matrix, this will take some time...Done, found 41080 words, 54732 words missing


Teraz wytrenujmy nasz model. Zauważ, że czas potrzebny na wytrenowanie modelu jest znacznie dłuższy niż w poprzednim przykładzie, ze względu na większy rozmiar warstwy osadzania, a co za tym idzie – znacznie większą liczbę parametrów. Ponadto, z tego powodu możemy potrzebować wytrenować nasz model na większej liczbie przykładów, jeśli chcemy uniknąć nadmiernego dopasowania.


In [12]:
train_epoch_emb(net,train_loader, lr=4, epoch_size=25000)

3200: acc=0.6359375
6400: acc=0.68109375
9600: acc=0.7067708333333333
12800: acc=0.723671875
16000: acc=0.73625
19200: acc=0.7463541666666667
22400: acc=0.7560714285714286


(214.1013875559821, 0.7626759436980166)

W naszym przypadku nie obserwujemy dużego wzrostu dokładności, co prawdopodobnie wynika z różnicy słowników. 
Aby pokonać problem różnych słowników, możemy skorzystać z jednego z następujących rozwiązań:
* Ponownie wytrenować model word2vec na naszym słowniku
* Załadować nasz zbiór danych ze słownikiem pochodzącym z wstępnie wytrenowanego modelu word2vec. Słownik używany do załadowania zbioru danych można określić podczas ładowania.

Drugi sposób wydaje się łatwiejszy, zwłaszcza że framework PyTorch `torchtext` zawiera wbudowane wsparcie dla embeddings. Możemy na przykład utworzyć słownik oparty na GloVe w następujący sposób:


In [14]:
vocab = torchtext.vocab.GloVe(name='6B', dim=50)

100%|█████████▉| 399999/400000 [00:15<00:00, 25411.14it/s]


Załadowany słownik ma następujące podstawowe operacje:
* `vocab.stoi` słownik pozwala nam zamienić słowo na jego indeks w słowniku
* `vocab.itos` robi odwrotnie - zamienia liczbę na słowo
* `vocab.vectors` jest tablicą wektorów osadzających, zatem aby uzyskać osadzenie słowa `s` musimy użyć `vocab.vectors[vocab.stoi[s]]`

Oto przykład manipulacji osadzeniami, aby zademonstrować równanie **kind-man+woman = queen** (musiałem trochę zmienić współczynnik, aby to zadziałało):


In [15]:
# get the vector corresponding to kind-man+woman
qvec = vocab.vectors[vocab.stoi['king']]-vocab.vectors[vocab.stoi['man']]+1.3*vocab.vectors[vocab.stoi['woman']]
# find the index of the closest embedding vector 
d = torch.sum((vocab.vectors-qvec)**2,dim=1)
min_idx = torch.argmin(d)
# find the corresponding word
vocab.itos[min_idx]

'queen'

Aby wytrenować klasyfikator z użyciem tych osadzeń, najpierw musimy zakodować nasz zbiór danych za pomocą słownictwa GloVe:


In [16]:
def offsetify(b):
    # first, compute data tensor from all sequences
    x = [torch.tensor(encode(t[1],voc=vocab)) for t in b] # pass the instance of vocab to encode function!
    # now, compute the offsets by accumulating the tensor of sequence lengths
    o = [0] + [len(t) for t in x]
    o = torch.tensor(o[:-1]).cumsum(dim=0)
    return ( 
        torch.LongTensor([t[0]-1 for t in b]), # labels
        torch.cat(x), # text 
        o
    )

Jak widzieliśmy powyżej, wszystkie wektorowe osadzenia są przechowywane w macierzy `vocab.vectors`. Sprawia to, że bardzo łatwo jest załadować te wagi do wag warstwy osadzającej za pomocą prostego kopiowania:


In [17]:
net = EmbedClassifier(len(vocab),len(vocab.vectors[0]),len(classes))
net.embedding.weight.data = vocab.vectors
net = net.to(device)

Teraz wytrenujmy nasz model i zobaczmy, czy uzyskamy lepsze wyniki:


In [18]:
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=16, collate_fn=offsetify, shuffle=True)
train_epoch_emb(net,train_loader, lr=4, epoch_size=25000)

3200: acc=0.6271875
6400: acc=0.68078125
9600: acc=0.7030208333333333
12800: acc=0.71984375
16000: acc=0.7346875
19200: acc=0.7455729166666667
22400: acc=0.7529464285714286


(35.53972978646833, 0.7575175943698017)

Jednym z powodów, dla których nie obserwujemy znaczącego wzrostu dokładności, jest fakt, że niektóre słowa z naszego zbioru danych nie występują w wstępnie wytrenowanym słowniku GloVe, a tym samym są zasadniczo pomijane. Aby pokonać ten problem, możemy wytrenować własne osadzenia na naszym zbiorze danych.


## Osadzenia kontekstowe

Jednym z kluczowych ograniczeń tradycyjnych, wstępnie wytrenowanych reprezentacji osadzeń, takich jak Word2Vec, jest problem rozróżniania znaczenia słów (ang. word sense disambiguation). Chociaż wstępnie wytrenowane osadzenia mogą uchwycić część znaczenia słów w kontekście, każde możliwe znaczenie słowa jest zakodowane w tym samym osadzeniu. Może to powodować problemy w modelach zależnych, ponieważ wiele słów, takich jak słowo „play”, ma różne znaczenia w zależności od kontekstu, w którym są używane.

Na przykład słowo „play” w dwóch poniższych zdaniach ma całkiem różne znaczenie:
- Poszedłem na **sztukę** do teatru.
- John chce się **bawić** z przyjaciółmi.

Podane powyżej wstępnie wytrenowane osadzenia reprezentują oba te znaczenia słowa „play” w tym samym osadzeniu. Aby pokonać to ograniczenie, musimy zbudować osadzenia bazujące na **modelu językowym**, który jest trenowany na dużym korpusie tekstu i *zna* sposób, w jaki słowa mogą być łączone w różnych kontekstach. Omówienie osadzeń kontekstowych wykracza poza zakres tego samouczka, ale wrócimy do nich, gdy będziemy mówić o modelach językowych w następnej jednostce.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Zastrzeżenie**:
Niniejszy dokument został przetłumaczony za pomocą usługi tłumaczenia AI [Co-op Translator](https://github.com/Azure/co-op-translator). Choć dążymy do dokładności, prosimy pamiętać, że automatyczne tłumaczenia mogą zawierać błędy lub niedokładności. Oryginalny dokument w jego języku źródłowym należy uznawać za autorytatywne źródło. W przypadku informacji krytycznych zalecane jest skorzystanie z profesjonalnego tłumaczenia wykonanego przez człowieka. Nie ponosimy odpowiedzialności za jakiekolwiek nieporozumienia lub błędne interpretacje wynikające z użycia tego tłumaczenia.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
